# Do columns separate the windows in combination that do not separate them alone?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


`TransactionDT` is deliberately not in the feature list. 

Include the timestamp and any discriminator becomes trivially perfect, which is where the folklore "adversarial AUC is 1 on this competition" partly comes from. The interesting question is what the *other* columns give away.

In [1]:
from pathlib import Path

import numpy as np
import polars as pl
from IPython.display import Markdown, display

from fraud_detection.evaluation.distribution_shift import (
    Reference,
    adversarial_auc,
    adversarial_per_feature,
)

C = [f"C{i}" for i in range(1, 15)]
D = [f"D{i}" for i in range(1, 16)]
M = [f"M{i}" for i in range(1, 10)]
V = [f"V{i}" for i in range(1, 340)]
FEATURES = C + D + M + V
DTYPE = {c: pl.Float32 for c in C + D + V}


def find(name):
    for p in (Path(f"../../input/ieee-fraud-detection/{name}"),
              Path(f"../../kaggle/raw/{name}"), Path(f"kaggle/raw/{name}")):
        if p.exists():
            return p
    raise FileNotFoundError(name)


train = pl.read_csv(find("train_transaction.csv"), columns=FEATURES, schema_overrides=DTYPE)
test = pl.read_csv(find("test_transaction.csv"), columns=FEATURES, schema_overrides=DTYPE)
print(f"train {len(train):,}   test {len(test):,}   features {len(FEATURES)}")

train 590,540   test 506,691   features 377


In [ ]:
ref = Reference.fit(train, FEATURES, meta={"source": "train_transaction.csv"})
psi = ref.psi(test)

In [ ]:
bands = pl.DataFrame({
    "Band": ["> 0.25 (significant)", "0.10 - 0.25", "<= 0.10", "degenerate"],
    "Columns": [
        int((psi["psi"] > 0.25).sum()),
        int(((psi["psi"] > 0.10) & (psi["psi"] <= 0.25)).sum()),
        int((psi["psi"] <= 0.10).sum()),
        int(psi["psi"].is_nan().sum())
    ]
})
Markdown(bands.to_pandas().to_markdown(index=False))


|    | Band                 |   Columns |
|---:|:---------------------|----------:|
|  0 | > 0.25 (significant) |       108 |
|  1 | 0.10 - 0.25          |        51 |
|  2 | <= 0.10              |       218 |
|  3 | degenerate           |        77 |

In [ ]:
df_psi = psi.select(["column", "psi", "null_share_ref", "null_share_cur"])

In [ ]:
display(Markdown(df_psi.to_pandas().to_markdown(index=False)))

| column   |           psi |   null_share_ref |   null_share_cur |
|:---------|--------------:|-----------------:|-----------------:|
| C3       | nan           |      0           |      5.92077e-06 |
| V1       | nan           |      0.472935    |      0.348374    |
| V14      | nan           |      0.128819    |      0.0248455   |
| V27      | nan           |      0.128819    |      0.0248455   |
| V28      | nan           |      0.128819    |      0.0248455   |
| V41      | nan           |      0.286126    |      0.151678    |
| V65      | nan           |      0.130552    |      0.0254573   |
| V68      | nan           |      0.130552    |      0.0254573   |
| V88      | nan           |      0.150987    |      0.0238429   |
| V89      | nan           |      0.150987    |      0.0238429   |
| V98      | nan           |      0.000531717 |      0           |
| V101     | nan           |      0.000531717 |      0           |
| V102     | nan           |      0.000531717 |      0           |
| V103     | nan           |      0.000531717 |      0           |
| V104     | nan           |      0.000531717 |      0           |
| V105     | nan           |      0.000531717 |      0           |
| V106     | nan           |      0.000531717 |      0           |
| V107     | nan           |      0.000531717 |      0           |
| V129     | nan           |      0.000531717 |      0           |
| V132     | nan           |      0.000531717 |      0           |
| V133     | nan           |      0.000531717 |      0           |
| V134     | nan           |      0.000531717 |      0           |
| V135     | nan           |      0.000531717 |      0           |
| V136     | nan           |      0.000531717 |      0           |
| V137     | nan           |      0.000531717 |      0           |
| V138     | nan           |      0.861237    |      0.850432    |
| V141     | nan           |      0.861237    |      0.850432    |
| V142     | nan           |      0.861237    |      0.850432    |
| V146     | nan           |      0.861237    |      0.850432    |
| V147     | nan           |      0.861237    |      0.850432    |
| V161     | nan           |      0.861237    |      0.850432    |
| V162     | nan           |      0.861237    |      0.850432    |
| V163     | nan           |      0.861237    |      0.850432    |
| V169     | nan           |      0.763235    |      0.730852    |
| V172     | nan           |      0.763554    |      0.730143    |
| V173     | nan           |      0.763554    |      0.730143    |
| V184     | nan           |      0.763235    |      0.730852    |
| V185     | nan           |      0.763235    |      0.730852    |
| V205     | nan           |      0.763554    |      0.730143    |
| V206     | nan           |      0.763554    |      0.730143    |
| V220     | nan           |      0.760531    |      0.728995    |
| V223     | nan           |      0.779134    |      0.749891    |
| V226     | nan           |      0.779134    |      0.749891    |
| V227     | nan           |      0.760531    |      0.728995    |
| V235     | nan           |      0.779134    |      0.749891    |
| V238     | nan           |      0.760531    |      0.728995    |
| V239     | nan           |      0.760531    |      0.728995    |
| V266     | nan           |      0.779134    |      0.749891    |
| V269     | nan           |      0.779134    |      0.749891    |
| V270     | nan           |      0.760531    |      0.728995    |
| V271     | nan           |      0.760531    |      0.728995    |
| V272     | nan           |      0.760531    |      0.728995    |
| V276     | nan           |      0.779134    |      0.749891    |
| V281     | nan           |      0.00214888  |      0.0119027   |
| V284     | nan           |      2.03204e-05 |      5.92077e-06 |
| V286     | nan           |      2.03204e-05 |      5.92077e-06 |
| V290     | nan           |      2.03204e-05 |      5.92077e-06 |
| V293     | nan           |      2.03204e-05 |      5.92077e-06 |
| V295     | nan           |      2.03204e-05 |      5.92077e-06 |
| V296     | nan           |      0.00214888  |      0.0119027   |
| V297     | nan           |      2.03204e-05 |      5.92077e-06 |
| V298     | nan           |      2.03204e-05 |      5.92077e-06 |
| V299     | nan           |      2.03204e-05 |      5.92077e-06 |
| V300     | nan           |      0.00214888  |      0.0119027   |
| V301     | nan           |      0.00214888  |      0.0119027   |
| V305     | nan           |      2.03204e-05 |      5.92077e-06 |
| V309     | nan           |      2.03204e-05 |      5.92077e-06 |
| V311     | nan           |      2.03204e-05 |      5.92077e-06 |
| V316     | nan           |      2.03204e-05 |      5.92077e-06 |
| V318     | nan           |      2.03204e-05 |      5.92077e-06 |
| V319     | nan           |      2.03204e-05 |      5.92077e-06 |
| V320     | nan           |      2.03204e-05 |      5.92077e-06 |
| V321     | nan           |      2.03204e-05 |      5.92077e-06 |
| V325     | nan           |      0.86055     |      0.849157    |
| V327     | nan           |      0.86055     |      0.849157    |
| V334     | nan           |      0.86055     |      0.849157    |
| V336     | nan           |      0.86055     |      0.849157    |
| V81      |   0.341498    |      0.150987    |      0.0238429   |
| V80      |   0.341318    |      0.150987    |      0.0238429   |
| D15      |   0.337931    |      0.150901    |      0.0238193   |
| V85      |   0.336594    |      0.150987    |      0.0238429   |
| V84      |   0.336095    |      0.150987    |      0.0238429   |
| V93      |   0.319722    |      0.150987    |      0.0238429   |
| V92      |   0.319655    |      0.150987    |      0.0238429   |
| V59      |   0.283374    |      0.130552    |      0.0254573   |
| V60      |   0.283032    |      0.130552    |      0.0254573   |
| V63      |   0.277639    |      0.130552    |      0.0254573   |
| V64      |   0.277454    |      0.130552    |      0.0254573   |
| V18      |   0.277263    |      0.128819    |      0.0248455   |
| V17      |   0.277215    |      0.128819    |      0.0248455   |
| V21      |   0.271148    |      0.128819    |      0.0248455   |
| V22      |   0.271144    |      0.128819    |      0.0248455   |
| V91      |   0.265824    |      0.150987    |      0.0238429   |
| V90      |   0.265756    |      0.150987    |      0.0238429   |
| V71      |   0.264084    |      0.130552    |      0.0254573   |
| V72      |   0.263537    |      0.130552    |      0.0254573   |
| V32      |   0.259787    |      0.128819    |      0.0248455   |
| V31      |   0.259709    |      0.128819    |      0.0248455   |
| V82      |   0.25521     |      0.150987    |      0.0238429   |
| V83      |   0.254368    |      0.150987    |      0.0238429   |
| V94      |   0.254228    |      0.150987    |      0.0238429   |
| V79      |   0.253393    |      0.150987    |      0.0238429   |
| V76      |   0.253162    |      0.150987    |      0.0238429   |
| V78      |   0.25309     |      0.150987    |      0.0238429   |
| V77      |   0.253075    |      0.150987    |      0.0238429   |
| V86      |   0.252936    |      0.150987    |      0.0238429   |
| V87      |   0.252895    |      0.150987    |      0.0238429   |
| V75      |   0.25248     |      0.150987    |      0.0238429   |
| V66      |   0.239684    |      0.130552    |      0.0254573   |
| V67      |   0.23429     |      0.130552    |      0.0254573   |
| V25      |   0.229483    |      0.128819    |      0.0248455   |
| V26      |   0.225618    |      0.128819    |      0.0248455   |
| D10      |   0.2146      |      0.128733    |      0.0247587   |
| V61      |   0.199492    |      0.130552    |      0.0254573   |
| V62      |   0.19653     |      0.130552    |      0.0254573   |
| V19      |   0.196306    |      0.128819    |      0.0248455   |
| V20      |   0.193716    |      0.128819    |      0.0248455   |
| V70      |   0.193696    |      0.130552    |      0.0254573   |
| V69      |   0.193291    |      0.130552    |      0.0254573   |
| V30      |   0.192931    |      0.128819    |      0.0248455   |
| V29      |   0.192555    |      0.128819    |      0.0248455   |
| V74      |   0.190116    |      0.130552    |      0.0254573   |
| V34      |   0.189289    |      0.128819    |      0.0248455   |
| V33      |   0.187095    |      0.128819    |      0.0248455   |
| V73      |   0.186794    |      0.130552    |      0.0254573   |
| V53      |   0.186507    |      0.130552    |      0.0254573   |
| V57      |   0.185579    |      0.130552    |      0.0254573   |
| V58      |   0.185485    |      0.130552    |      0.0254573   |
| V16      |   0.185478    |      0.128819    |      0.0248455   |
| V15      |   0.185476    |      0.128819    |      0.0248455   |
| V54      |   0.184427    |      0.130552    |      0.0254573   |
| V56      |   0.183814    |      0.130552    |      0.0254573   |
| V55      |   0.183811    |      0.130552    |      0.0254573   |
| V12      |   0.183261    |      0.128819    |      0.0248455   |
| V23      |   0.182995    |      0.128819    |      0.0248455   |
| V24      |   0.182986    |      0.128819    |      0.0248455   |
| V13      |   0.182836    |      0.128819    |      0.0248455   |
| V40      |   0.17927     |      0.286126    |      0.151678    |
| V39      |   0.179141    |      0.286126    |      0.151678    |
| V43      |   0.174971    |      0.286126    |      0.151678    |
| V42      |   0.174683    |      0.286126    |      0.151678    |
| V50      |   0.160515    |      0.286126    |      0.151678    |
| D4       |   0.147966    |      0.286047    |      0.151672    |
| D13      |   0.141464    |      0.895093    |      0.756491    |
| D11      |   0.128615    |      0.472935    |      0.348374    |
| V49      |   0.118195    |      0.286126    |      0.151678    |
| V48      |   0.118155    |      0.286126    |      0.151678    |
| D14      |   0.115412    |      0.894695    |      0.772654    |
| V52      |   0.110542    |      0.286126    |      0.151678    |
| V51      |   0.109525    |      0.286126    |      0.151678    |
| V44      |   0.108886    |      0.286126    |      0.151678    |
| V45      |   0.108841    |      0.286126    |      0.151678    |
| V47      |   0.108776    |      0.286126    |      0.151678    |
| V46      |   0.108771    |      0.286126    |      0.151678    |
| V38      |   0.108679    |      0.286126    |      0.151678    |
| V37      |   0.108669    |      0.286126    |      0.151678    |
| V36      |   0.108608    |      0.286126    |      0.151678    |
| V35      |   0.108579    |      0.286126    |      0.151678    |
| D6       |   0.108132    |      0.876068    |      0.75373     |
| C13      |   0.086429    |      0           |      0.0093706   |
| V4       |   0.064767    |      0.472935    |      0.348374    |
| V5       |   0.064666    |      0.472935    |      0.348374    |
| V10      |   0.0646654   |      0.472935    |      0.348374    |
| V11      |   0.064638    |      0.472935    |      0.348374    |
| V8       |   0.0645838   |      0.472935    |      0.348374    |
| V9       |   0.0645838   |      0.472935    |      0.348374    |
| V6       |   0.0645674   |      0.472935    |      0.348374    |
| V7       |   0.0645674   |      0.472935    |      0.348374    |
| V2       |   0.0645237   |      0.472935    |      0.348374    |
| V3       |   0.0645237   |      0.472935    |      0.348374    |
| C12      |   0.0635895   |      0           |      5.92077e-06 |
| M9       |   0.0634879   |      0.586331    |      0.463801    |
| M8       |   0.0610144   |      0.586331    |      0.463801    |
| M7       |   0.0606937   |      0.586353    |      0.463829    |
| M2       |   0.0549333   |      0.459071    |      0.348613    |
| M3       |   0.0523252   |      0.459071    |      0.348613    |
| M1       |   0.0509297   |      0.459071    |      0.348613    |
| D7       |   0.0357239   |      0.934099    |      0.881322    |
| D5       |   0.0315936   |      0.524674    |      0.442824    |
| C2       |   0.0226913   |      0           |      5.92077e-06 |
| C1       |   0.0225596   |      0           |      5.92077e-06 |
| D1       |   0.021302    |      0.00214888  |      0.0119027   |
| V283     |   0.0192438   |      0.00214888  |      0.0119027   |
| V282     |   0.0185874   |      0.00214888  |      0.0119027   |
| V314     |   0.0170127   |      0.00214888  |      0.0119027   |
| V315     |   0.0168957   |      0.00214888  |      0.0119027   |
| V313     |   0.0168387   |      0.00214888  |      0.0119027   |
| V288     |   0.0168042   |      0.00214888  |      0.0119027   |
| V289     |   0.0167962   |      0.00214888  |      0.0119027   |
| C11      |   0.0116249   |      0           |      5.92077e-06 |
| D3       |   0.0108948   |      0.445149    |      0.400919    |
| V150     |   0.0102016   |      0.861227    |      0.849899    |
| D12      |   0.00968996  |      0.89041     |      0.863321    |
| D2       |   0.00794716  |      0.475492    |      0.463338    |
| C10      |   0.00680759  |      0           |      5.92077e-06 |
| V222     |   0.00651372  |      0.760531    |      0.728995    |
| V176     |   0.00639927  |      0.763554    |      0.730143    |
| V211     |   0.00630968  |      0.763554    |      0.730143    |
| V221     |   0.00630795  |      0.760531    |      0.728995    |
| V216     |   0.00624006  |      0.763554    |      0.730143    |
| V215     |   0.00619068  |      0.763554    |      0.730143    |
| V251     |   0.00618203  |      0.760531    |      0.728995    |
| V256     |   0.00618203  |      0.760531    |      0.728995    |
| V245     |   0.00617655  |      0.760531    |      0.728995    |
| V250     |   0.00617655  |      0.760531    |      0.728995    |
| V255     |   0.00617655  |      0.760531    |      0.728995    |
| V259     |   0.00617655  |      0.760531    |      0.728995    |
| V213     |   0.00613548  |      0.763554    |      0.730143    |
| V204     |   0.00611378  |      0.763554    |      0.730143    |
| V171     |   0.00610045  |      0.763235    |      0.730852    |
| V170     |   0.00609454  |      0.763235    |      0.730852    |
| V212     |   0.00607922  |      0.763554    |      0.730143    |
| V168     |   0.00605805  |      0.763554    |      0.730143    |
| V207     |   0.00605502  |      0.763554    |      0.730143    |
| V214     |   0.00602823  |      0.763554    |      0.730143    |
| V202     |   0.00601519  |      0.763554    |      0.730143    |
| V181     |   0.00599052  |      0.763554    |      0.730143    |
| V199     |   0.00599029  |      0.763554    |      0.730143    |
| V186     |   0.00598998  |      0.763554    |      0.730143    |
| V190     |   0.00598998  |      0.763554    |      0.730143    |
| V191     |   0.00598998  |      0.763554    |      0.730143    |
| V196     |   0.00598998  |      0.763554    |      0.730143    |
| V189     |   0.00598961  |      0.763235    |      0.730852    |
| V195     |   0.00598961  |      0.763235    |      0.730852    |
| V198     |   0.00598961  |      0.763235    |      0.730852    |
| V201     |   0.00598961  |      0.763235    |      0.730852    |
| V188     |   0.00598512  |      0.763235    |      0.730852    |
| V194     |   0.00598512  |      0.763235    |      0.730852    |
| V197     |   0.00598512  |      0.763235    |      0.730852    |
| V200     |   0.00598512  |      0.763235    |      0.730852    |
| V183     |   0.00598164  |      0.763554    |      0.730143    |
| V193     |   0.00597704  |      0.763554    |      0.730143    |
| V177     |   0.00595072  |      0.763554    |      0.730143    |
| V203     |   0.00594603  |      0.763554    |      0.730143    |
| V167     |   0.0059415   |      0.763554    |      0.730143    |
| V182     |   0.00593643  |      0.763554    |      0.730143    |
| V179     |   0.00592915  |      0.763554    |      0.730143    |
| V302     |   0.00592741  |      2.03204e-05 |      5.92077e-06 |
| V187     |   0.00592201  |      0.763554    |      0.730143    |
| V192     |   0.00592201  |      0.763554    |      0.730143    |
| V178     |   0.00591168  |      0.763554    |      0.730143    |
| V304     |   0.00590186  |      2.03204e-05 |      5.92077e-06 |
| V303     |   0.00587839  |      2.03204e-05 |      5.92077e-06 |
| V175     |   0.00578525  |      0.763235    |      0.730852    |
| V174     |   0.00570684  |      0.763235    |      0.730852    |
| D8       |   0.00566199  |      0.873123    |      0.853287    |
| V209     |   0.00563899  |      0.763235    |      0.730852    |
| V180     |   0.00558889  |      0.763235    |      0.730852    |
| V208     |   0.00557932  |      0.763235    |      0.730852    |
| V210     |   0.00557075  |      0.763235    |      0.730852    |
| M4       |   0.00550715  |      0.476588    |      0.469211    |
| V234     |   0.0054653   |      0.760531    |      0.728995    |
| V127     |   0.00545118  |      0.000531717 |      0           |
| C4       |   0.00542095  |      0           |      5.92077e-06 |
| V264     |   0.00522551  |      0.779134    |      0.749891    |
| V265     |   0.00515207  |      0.779134    |      0.749891    |
| V230     |   0.00512794  |      0.779134    |      0.749891    |
| V229     |   0.00512343  |      0.779134    |      0.749891    |
| V114     |   0.005114    |      0.000531717 |      0           |
| V123     |   0.005114    |      0.000531717 |      0           |
| V108     |   0.0051051   |      0.000531717 |      0           |
| V111     |   0.0051051   |      0.000531717 |      0           |
| V117     |   0.0051051   |      0.000531717 |      0           |
| V120     |   0.0051051   |      0.000531717 |      0           |
| V228     |   0.00504124  |      0.779134    |      0.749891    |
| V218     |   0.00500345  |      0.779134    |      0.749891    |
| V263     |   0.00497416  |      0.779134    |      0.749891    |
| V219     |   0.00497001  |      0.779134    |      0.749891    |
| V217     |   0.00490655  |      0.779134    |      0.749891    |
| C8       |   0.00488924  |      0           |      5.92077e-06 |
| V267     |   0.00481805  |      0.779134    |      0.749891    |
| V143     |   0.00481594  |      0.861227    |      0.849899    |
| V130     |   0.00480743  |      0.000531717 |      0           |
| V273     |   0.0048042   |      0.779134    |      0.749891    |
| V236     |   0.00480179  |      0.779134    |      0.749891    |
| V237     |   0.00479018  |      0.779134    |      0.749891    |
| V257     |   0.00478036  |      0.779134    |      0.749891    |
| V242     |   0.00478022  |      0.779134    |      0.749891    |
| V244     |   0.00478022  |      0.779134    |      0.749891    |
| V246     |   0.00478022  |      0.779134    |      0.749891    |
| V247     |   0.00478022  |      0.779134    |      0.749891    |
| V252     |   0.00478022  |      0.779134    |      0.749891    |
| V241     |   0.00477877  |      0.779134    |      0.749891    |
| V274     |   0.00477639  |      0.779134    |      0.749891    |
| V275     |   0.00477452  |      0.779134    |      0.749891    |
| V268     |   0.00477079  |      0.779134    |      0.749891    |
| V258     |   0.00477054  |      0.779134    |      0.749891    |
| V243     |   0.00477047  |      0.779134    |      0.749891    |
| V249     |   0.00477047  |      0.779134    |      0.749891    |
| V254     |   0.00477047  |      0.779134    |      0.749891    |
| V224     |   0.00477028  |      0.779134    |      0.749891    |
| V225     |   0.0047657   |      0.779134    |      0.749891    |
| V240     |   0.00476286  |      0.779134    |      0.749891    |
| V260     |   0.00475978  |      0.779134    |      0.749891    |
| V231     |   0.00475949  |      0.779134    |      0.749891    |
| V261     |   0.00475679  |      0.779134    |      0.749891    |
| V233     |   0.00475658  |      0.779134    |      0.749891    |
| V248     |   0.00475608  |      0.779134    |      0.749891    |
| V253     |   0.00475608  |      0.779134    |      0.749891    |
| V262     |   0.00475595  |      0.779134    |      0.749891    |
| V277     |   0.00475541  |      0.779134    |      0.749891    |
| V232     |   0.00475496  |      0.779134    |      0.749891    |
| V278     |   0.00475493  |      0.779134    |      0.749891    |
| V96      |   0.00461699  |      0.000531717 |      0           |
| V99      |   0.00454091  |      0.000531717 |      0           |
| V116     |   0.00446793  |      0.000531717 |      0           |
| V125     |   0.00446793  |      0.000531717 |      0           |
| V110     |   0.00445965  |      0.000531717 |      0           |
| V113     |   0.00445965  |      0.000531717 |      0           |
| V119     |   0.00445965  |      0.000531717 |      0           |
| V122     |   0.00445965  |      0.000531717 |      0           |
| C7       |   0.0041515   |      0           |      5.92077e-06 |
| V128     |   0.00407254  |      0.000531717 |      0           |
| C5       |   0.00399337  |      0           |      5.92077e-06 |
| V97      |   0.00379032  |      0.000531717 |      0           |
| M6       |   0.00372098  |      0.286788    |      0.31368     |
| V115     |   0.00360964  |      0.000531717 |      0           |
| V124     |   0.00360964  |      0.000531717 |      0           |
| V109     |   0.00360321  |      0.000531717 |      0           |
| V112     |   0.00360321  |      0.000531717 |      0           |
| V118     |   0.00360321  |      0.000531717 |      0           |
| V121     |   0.00360321  |      0.000531717 |      0           |
| V131     |   0.00356127  |      0.000531717 |      0           |
| D9       |   0.00348222  |      0.873123    |      0.853287    |
| V95      |   0.0034349   |      0.000531717 |      0           |
| C14      |   0.00340176  |      0           |      5.92077e-06 |
| V126     |   0.00337084  |      0.000531717 |      0           |
| V100     |   0.00333487  |      0.000531717 |      0           |
| V164     |   0.00321071  |      0.861227    |      0.849899    |
| V145     |   0.00299244  |      0.861227    |      0.849899    |
| V317     |   0.00275888  |      2.03204e-05 |      5.92077e-06 |
| V165     |   0.00253966  |      0.861227    |      0.849899    |
| V166     |   0.0023728   |      0.861227    |      0.849899    |
| V152     |   0.00232681  |      0.861227    |      0.849899    |
| V144     |   0.00201805  |      0.861227    |      0.849899    |
| V294     |   0.00198081  |      2.03204e-05 |      5.92077e-06 |
| C6       |   0.00161801  |      0           |      5.92077e-06 |
| V160     |   0.0014743   |      0.861227    |      0.849899    |
| V333     |   0.00142858  |      0.86055     |      0.849157    |
| M5       |   0.00134183  |      0.593494    |      0.611086    |
| V324     |   0.00134077  |      0.86055     |      0.849157    |
| V322     |   0.00127279  |      0.86055     |      0.849157    |
| V323     |   0.00125511  |      0.86055     |      0.849157    |
| V139     |   0.00124584  |      0.861237    |      0.850432    |
| V140     |   0.00121433  |      0.861237    |      0.850432    |
| V326     |   0.0011925   |      0.86055     |      0.849157    |
| V151     |   0.00119137  |      0.861227    |      0.849899    |
| V335     |   0.00118369  |      0.86055     |      0.849157    |
| V332     |   0.0011651   |      0.86055     |      0.849157    |
| V328     |   0.00105369  |      0.86055     |      0.849157    |
| V331     |   0.00105299  |      0.86055     |      0.849157    |
| V338     |   0.00105158  |      0.86055     |      0.849157    |
| V337     |   0.00104914  |      0.86055     |      0.849157    |
| V330     |   0.00104909  |      0.86055     |      0.849157    |
| V329     |   0.00104726  |      0.86055     |      0.849157    |
| V339     |   0.00104669  |      0.86055     |      0.849157    |
| V159     |   0.00103918  |      0.861227    |      0.849899    |
| V149     |   0.000964165 |      0.861237    |      0.850432    |
| V154     |   0.000964165 |      0.861237    |      0.850432    |
| V156     |   0.000964165 |      0.861237    |      0.850432    |
| V158     |   0.000964165 |      0.861237    |      0.850432    |
| V148     |   0.000963612 |      0.861237    |      0.850432    |
| V153     |   0.000963612 |      0.861237    |      0.850432    |
| V155     |   0.000963612 |      0.861237    |      0.850432    |
| V157     |   0.000963612 |      0.861237    |      0.850432    |
| C9       |   0.000929939 |      0           |      5.92077e-06 |
| V280     |   0.000822086 |      2.03204e-05 |      5.92077e-06 |
| V291     |   0.000758652 |      2.03204e-05 |      5.92077e-06 |
| V279     |   0.000372319 |      2.03204e-05 |      5.92077e-06 |
| V310     |   0.00035861  |      2.03204e-05 |      5.92077e-06 |
| V307     |   0.000300282 |      2.03204e-05 |      5.92077e-06 |
| V292     |   0.00029691  |      2.03204e-05 |      5.92077e-06 |
| V308     |   0.000276991 |      2.03204e-05 |      5.92077e-06 |
| V312     |   0.000216833 |      2.03204e-05 |      5.92077e-06 |
| V287     |   9.27849e-05 |      2.03204e-05 |      5.92077e-06 |
| V285     |   6.83751e-05 |      2.03204e-05 |      5.92077e-06 |
| V306     |   3.33091e-05 |      2.03204e-05 |      5.92077e-06 |

In [ ]:
rng = np.random.default_rng(0)
N = 100_000
tr_s = train[rng.choice(len(train), N, replace=False)]
te_s = test[rng.choice(len(test), N, replace=False)]

auc, importance = adversarial_auc(tr_s, te_s, FEATURES)
print(f"out-of-fold AUC over all {len(FEATURES)} columns: {auc:.4f}")
Markdown(importance.head(8).to_pandas().to_markdown(index=False))


KeyboardInterrupt: 

In [ ]:
per_feature = adversarial_per_feature(tr_s, te_s, FEATURES, n_jobs=-1)
Markdown(per_feature.head(8).to_pandas().to_markdown(index=False))


## Combinatorial Separation

The best single column is `D15` at 0.656. The rest fall away quickly. Yet all 377 together reach 0.838.

The separation lives in the *combination*, not in any marginal. A drift monitor that iterates columns and alerts on per-column thresholds will not see it — not because the thresholds are set wrong, but because the signal is not in the quantity being measured.